In [ ]:
# -*- coding: utf-8 -*-

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import Row

"""--------------------------------------------------------------------------
dataframe-based decision tree learning
-------------------------------------------------------------------------"""

# stop SpSession.stop() if exist
# Robustly stop any existing SparkContext
try:
    # Try to stop the specific variable if it exists
    if 'SpContext' in locals() or 'SpContext' in globals():
        SpContext.stop()
    # Also try the general method to catch any other contexts
    SparkContext.getOrCreate().stop()
except:
    pass

print("=== starting dataframe-based decision tree learning ===")
# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("DecisionTreeExample")
SpContext = SparkContext(conf = conf)

# Create a SparkSession (the config bit is only for Windows!)
SpSession = SparkSession.builder.config("spark.sql.warehouse.dir", "file:///C:/temp").getOrCreate()

"""--------------------------------------------------------------------------
Load Data
-------------------------------------------------------------------------"""
# Use the raw URL to download the CSV content, not the HTML page
!rm -f 8-iris.csv
!wget -O 8-iris.csv https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-iris.csv

print("=== start loading data ===")
#Load the CSV file into a RDD
irisData = SpContext.textFile("8-iris.csv")
irisData.cache()


#Remove the first line (contains headers)
dataLines = irisData.filter(lambda x: "Sepal" not in x)
# Also filter out empty lines to avoid ValueError during conversion
dataLines = dataLines.filter(lambda x: len(x) > 0)
print("Total count of data: "+ str(dataLines.count()))

"""--------------------------------------------------------------------------
Cleanup and read data into mamory
-------------------------------------------------------------------------"""
print("=== start reading data ===")
from pyspark.sql import Row
#Create a Data Frame from the data
parts = dataLines.map(lambda l: l.split(","))
# Add a check to ensure the line has enough parts before converting
irisMap = parts.filter(lambda p: len(p) >= 5).map(lambda p: Row(SEPAL_LENGTH=float(p[0]),\
                                SEPAL_WIDTH=float(p[1]), \
                                PETAL_LENGTH=float(p[2]), \
                                PETAL_WIDTH=float(p[3]), \
                                SPECIES=p[4] ))

# Infer the schema, and register the DataFrame as a table.
irisDf = SpSession.createDataFrame(irisMap)
irisDf.cache()

#Add a numeric indexer for the label/target column
#Why? because the algorithm cannot handle string names
from pyspark.ml.feature import StringIndexer
stringIndexer = StringIndexer(inputCol="SPECIES", outputCol="IND_SPECIES")
si_model = stringIndexer.fit(irisDf)
irisNormDf = si_model.transform(irisDf)

irisNormDf.select("SPECIES","IND_SPECIES").distinct().collect()
irisNormDf.cache()

irisNormDf.describe().show(10)


"""--------------------------------------------------------------------------
Perform Data Analytics
-------------------------------------------------------------------------"""
print("=== start data analysis ===")

#Find correlation between predictors and target
for i in irisNormDf.columns:
    if not( isinstance(irisNormDf.select(i).take(1)[0][0], str)) :
        print( "Correlation to Species for ", i, \
                    irisNormDf.stat.corr('IND_SPECIES',i))

print("irisNormDf.select('SPECIES'): "+str(irisNormDf.select("SPECIES")));#return a dataframe contains the species column
print("irisNormDf.select('SPECIES').take(1): "+str(irisNormDf.select("SPECIES").take(1))); # return Row[], row list
print("irisNormDf.select('SPECIES').take(1)[0]: "+str(irisNormDf.select("SPECIES").take(1)[0])); #return the first row Row[0] boject
print("irisNormDf.select('SPECIES').take(1)[0][0]: "+irisNormDf.select("SPECIES").take(1)[0][0]); #return the first element of Row


"""--------------------------------------------------------------------------
Prepare data for ML
-------------------------------------------------------------------------"""
print("=== start prepare data for ML ===")
#Transform to a Data Frame for input to Machine Learing
#Drop columns that are not required (low correlation)

from pyspark.ml.linalg import Vectors
def transformToLabeledPoint(row) :
  # lp = label point
  # changing data vectors into label points
    lp = (row["SPECIES"], row["IND_SPECIES"], \
            Vectors.dense(
                [
                    row["SEPAL_LENGTH"],\
                    row["SEPAL_WIDTH"], \
                    row["PETAL_LENGTH"], \
                    row["PETAL_WIDTH"]
                ]
            )
        )
    return lp

irisLp = irisNormDf.rdd.map(transformToLabeledPoint)
irisLpDf = SpSession.createDataFrame(irisLp,["species","label", "features"])
irisLpDf.select("species","label","features").orderBy("features").show()
irisLpDf.cache()


"""--------------------------------------------------------------------------
Perform Machine Learning
-------------------------------------------------------------------------"""
print("=== start building model ===")
#Split into training and testing data
(trainingData, testData) = irisLpDf.randomSplit([0.8, 0.2])
trainingData.count()
testData.count()
testData.collect()

#import the decision tree algorithm class and its evaluator
from pyspark.ml.classification import DecisionTreeClassifier
# pyspark.ml - dataframe algorithm
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Create the model
dtClassifer = DecisionTreeClassifier(impurity='gini', maxDepth=2, labelCol="label",featuresCol="features")
# maxDepth decides how complicated decision tree would be
dtModel = dtClassifer.fit(trainingData)



"""--------------------------------------------------------------------------
Print out learned model
-------------------------------------------------------------------------"""
print("=== print out learned model ===")


nodeCount = dtModel.numNodes
depth = dtModel.depth
treeRules = dtModel

print("nodeCount: "+str(nodeCount)) #show how many nodes were learned
print("depth: "+str(depth)) #show the depth of the tree
print(str(treeRules.toDebugString)) #show the learned tree as rules



"""--------------------------------------------------------------------------
Evaluate learned model
-------------------------------------------------------------------------"""
print("=== print evaluation reults ===")

#Predict on the test data
predictions = dtModel.transform(testData)
predictions.select("prediction","species","label").collect()

#Evaluate precision
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction", \
                    labelCol="label",metricName="weightedPrecision")
precision = evaluator.evaluate(predictions)
print("The precision of the model is: "+str(precision))


#Draw a confusion matrix
predictions.groupBy("label","prediction").count().show()

SpSession.stop()

=== starting dataframe-based decision tree learning ===
--2026-03-01 18:51:35--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-iris.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3867 (3.8K) [text/plain]
Saving to: ‘8-iris.csv’

8-iris.csv          100%[===================>]   3.78K  --.-KB/s    in 0s      

2026-03-01 18:51:35 (40.1 MB/s) - ‘8-iris.csv’ saved [3867/3867]

=== start loading data ===
Total count of data: 150
=== start reading data ===
+-------+------------------+-------------------+------------------+------------------+---------+------------------+
|summary|      SEPAL_LENGTH|        SEPAL_WIDTH|      PETAL_LENGTH|       PETAL_WIDTH|  SPECIES|       IND_SPECIES|
+-------+------------------+-------------------+-----

In [ ]:
from pyspark import SparkConf, SparkContext
from pyspark.mllib.tree import DecisionTree, DecisionTreeModel
from pyspark.mllib.util import MLUtils

"""--------------------------------------------------------------------------
RDD-based decision tree learning
-------------------------------------------------------------------------"""
# Robustly stop any existing SparkContext
try:
    if 'SpContext' in locals() or 'SpContext' in globals():
        SpContext.stop()
    SparkContext.getOrCreate().stop()
except:
    pass

print("=== starting RDD-based decision tree learning ===")

# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("DecisionTreeDiabetesExample") # treat every core of your desktop as an executor
SpContext = SparkContext(conf = conf)

# Load and parse the data file (in SVM format ) into an RDD of LabeledPoint.
# Use raw URL to avoid downloading HTML
!rm -f 8-diabetes.txt
!wget -O 8-diabetes.txt https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-diabetes.txt

data = MLUtils.loadLibSVMFile(SpContext, '8-diabetes.txt')
# Split the data into training and test sets (20% held out for testing)
(trainingDataSet, testDataSet) = data.randomSplit([0.8, 0.2])

# Train a DecisionTree model.
#  Empty categoricalFeaturesInfo indicates all features are continuous.
model = DecisionTree.trainClassifier(trainingDataSet, impurity='gini', numClasses=2, maxDepth=8, maxBins=100, categoricalFeaturesInfo={})

#print the learned model
print('Learned classification tree model:')
print(str(model.toDebugString()))

# Evaluate model on test instances and compute test error
predictions = model.predict(testDataSet.map(lambda testData: testData.features))
labelsAndPredictions = testDataSet.map(lambda testData: testData.label).zip(predictions)
testErr = labelsAndPredictions.filter(lambda pair: pair[0] != pair[1]).count() / float(testDataSet.count())
print('Test Error Rate = ' + str(testErr))
print('Test Accuracy = '+ str(1-testErr))

=== starting RDD-based decision tree learning ===
--2026-03-01 18:52:54--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-diabetes.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 75501 (74K) [text/plain]
Saving to: ‘8-diabetes.txt’

8-diabetes.txt      100%[===================>]  73.73K  --.-KB/s    in 0.01s   

2026-03-01 18:52:54 (5.92 MB/s) - ‘8-diabetes.txt’ saved [75501/75501]

Learned classification tree model:
DecisionTreeModel classifier of depth 8 with 147 nodes
  If (feature 1 <= 143.5)
   If (feature 7 <= 28.5)
    If (feature 1 <= 127.5)
     If (feature 5 <= 45.3500005)
      If (feature 5 <= 30.8499995)
       If (feature 6 <= 0.673)
        Predict: 1.0
       Else (feature 6 > 0.673)
        If (feature 6 <= 0.684)
 